In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from algorithm_evaluation import evaluate

# ---- LaTeX rendering for plots (Computer Modern, matches paper) ----
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}",
})

# The sweeps take hours. Leave False to load the shipped results from
# algorithm_evaluation/results/; set True to regenerate and overwrite them.
RECOMPUTE = False

## Simulation sweep


In [ ]:
N_values = [10, 15, 20, 30]
c_values = [0.5, 1.0, 1.5, 2.0]
N_SIMS   = 100    # 100
EXP_SIZE = 200    # 200
SEED     = 42

theta_config = {
    'CF_fraction': 1.0,   # always a 2-world counterfactual query
    'W_sizes': [1],
    'Z_sizes': [1],
}

experiment_config = {
    'CF_fraction': 0.3,           
    'W_sizes': [1, 2, 3],
    'Z_sizes': [1, 2, 3],
}

if RECOMPUTE:
    rows = []
    for N in N_values:
        p = 0.5 * np.log(N) / N       
        for c in c_values:
            q = c / N
            print(f"N={N:2d}, c={c:.1f}  (p={p:.3f}, q={q:.4f}) ...", end=" ", flush=True)
            df_chunk = evaluate(
                graph_source="erdos_renyi",
                graph_config={"N": N, "p": p, "q": q},
                n_simulations=N_SIMS,
                seed=SEED,
                experiment_set_size=EXP_SIZE,
                theta_config=theta_config,
                experiment_config=experiment_config,
            )
            df_chunk["c"] = c
            rows.append(df_chunk)
            total = df_chunk["n_experiments"].sum()
            confounded = df_chunk[df_chunk["n_R_theta_star"] > 0]
            bl_cond = (confounded["blocked_pruned"].sum() / confounded["n_experiments"].sum()
                       if len(confounded) > 0 else float("nan"))
            print(
                f"pruned={df_chunk['fraction_pruned'].mean():.0%}  "
                f"blocked={df_chunk['blocked_pruned'].sum()/total:.0%}  "
                f"|U|/|V|={df_chunk['u_v_ratio'].mean():.2f}  "
                f"pct_confounded={len(confounded)/len(df_chunk):.0%}  "
                f"blocked|confounded={bl_cond:.0%}"
            )

    df = pd.concat(rows, ignore_index=True)
    df.to_csv("algorithm_evaluation/results/er.csv", index=False)
    print(f"\nTotal simulations: {len(df)}")

In [ ]:
# Written above when RECOMPUTE is True; otherwise the shipped results.
df = pd.read_csv("algorithm_evaluation/results/er.csv")
N_values = [10, 15, 20, 30]

In [ ]:
def agg_group(g):
    total = g["n_experiments"].sum()
    conf  = g[g["n_R_theta_star"] > 0]
    return pd.Series({
        "mean_uv":         g["u_v_ratio"].mean(),          # actual |U|/|V|, used as x-axis
        "frac_blocked":    g["blocked_pruned"].sum() / total,
        "frac_id":         g["id_pruned"].sum()      / total,
        "frac_pruned":     g["fraction_pruned"].mean(),
        "pct_confounded":  len(conf) / len(g),
        "blocked_cond":    (conf["blocked_pruned"].sum() / conf["n_experiments"].sum()
                            if len(conf) > 0 else float("nan")),
        "avg_districts":   g["n_districts"].mean(),
    })

agg = df.groupby(["n_vars", "c"]).apply(agg_group).reset_index()
print(agg)

# Paper plots

In [ ]:
# ---- tunable font sizes ----
LABEL_FONTSIZE  = 12
TICK_FONTSIZE   = 11
LEGEND_FONTSIZE = 11
TITLE_FONTSIZE  = 13
HATCH_BLOCKED   = '///'   # B&W-readable pattern for AllPathsIntercepted
# ----------------------------

SHOW_BEFORE_STAR = False   

fig, axes = plt.subplots(1, len(N_values), figsize=(14, 3), sharey=True)

C_STAR = 0.5   

for ax, N in zip(axes, N_values):
    sub = agg[agg["n_vars"] == N].sort_values("mean_uv")
    uv_star = sub.loc[sub["c"] == C_STAR, "mean_uv"].values[0]

    if SHOW_BEFORE_STAR:
        # Pad at x=0 so the filled area starts at the origin
        xs      = np.concatenate([[0], sub["mean_uv"].values])
        blocked = np.concatenate([[sub["frac_blocked"].values[0]], sub["frac_blocked"].values])
        id_     = np.concatenate([[sub["frac_id"].values[0]],      sub["frac_id"].values])
        x_min   = 0
    else:
        # Only show data from uv_star onward
        mask    = sub["mean_uv"] >= uv_star
        xs      = sub.loc[mask, "mean_uv"].values
        blocked = sub.loc[mask, "frac_blocked"].values
        id_     = sub.loc[mask, "frac_id"].values
        x_min   = uv_star

    polys = ax.stackplot(
        xs, blocked, id_,
        labels=["AllPathsIntercepted", "ID check"],
        colors=["#DD8452", "#4C72B0"],
        alpha=0.88,
        edgecolor="black",
        linewidth=0.6,
    )
    # Hatch only the bottom layer (AllPathsIntercepted) for B&W readability
    polys[0].set_hatch(HATCH_BLOCKED)

    # ax.axvline(uv_star, color="black", lw=1.2, ls="--", alpha=0.6)
    # ax.text(uv_star + 0.01, 0.97, f"|U|/|V|={uv_star:.2f}",
    #         ha="left", va="top", fontsize=7.5, transform=ax.get_xaxis_transform())

    ax.set_title(f"$|\mathbf{{V}}|$ = {N}", fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(r"$|\mathbf{U}|/|\mathbf{V}|$", fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.set_ylim(0, 1)
    ax.set_xlim(x_min, xs[-1])
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}"))

axes[0].set_ylabel("Fraction of experiments pruned", fontsize=LABEL_FONTSIZE)

handles, labels = axes[0].get_legend_handles_labels()
# fig.legend(handles, labels, loc="upper right", fontsize=LEGEND_FONTSIZE, framealpha=0.9)
# fig.suptitle(
#     r"Pruning breakdown vs confounder density",
#     y=1.02, fontsize=TITLE_FONTSIZE
# )
plt.tight_layout()
plt.savefig("plots/er_fraction_pruned.png", bbox_inches="tight", dpi=300)
plt.show()
print(df.shape[0], "total simulations")

In [ ]:
# Overall stats
frac_id = df['id_pruned'] / df['n_experiments']
frac_bl = df['blocked_pruned'] / df['n_experiments']

print("=== Overall ===")
print(f"Total simulations: {len(df)}")
print(f"Mean total pruning:        {df['fraction_pruned'].mean():.3f} (std {df['fraction_pruned'].std():.3f})")
print(f"Mean ID pruning:           {frac_id.mean():.3f} (std {frac_id.std():.3f})")
print(f"Mean interception pruning: {frac_bl.mean():.3f} (std {frac_bl.std():.3f})")
print(f"Max total pruning (sim):   {df['fraction_pruned'].max():.3f}")

# Per c-density (sparsest -> densest)
per_c = (
    df.assign(frac_bl=frac_bl, frac_id=frac_id)
      .groupby('c')
      .agg(uv_mean=('u_v_ratio','mean'),
           bl_mean=('frac_bl','mean'),
           bl_std=('frac_bl','std'),
           id_mean=('frac_id','mean'),
           id_std=('frac_id','std'),
           total_mean=('fraction_pruned','mean'),
           total_std=('fraction_pruned','std'))
      .round(3)
)
print("\nPer-density (c) breakdown:")
print(per_c.to_string())
